&emsp;&emsp;不过需要注意的是，LangGraph是基于LangChain进行的构建，**无论图结构多复杂，单独每个任务执行链路仍然是现行的，其背后仍然是靠着LangChain的Chain来实现的**。因此我们可以这么来描述LangChain和LangGraph之间的关系，**LangGraph是LangChain工作流的高级编排工具**，其中“高级”之处就是LangGraph能按照图结构来编排工作流。

### 1. LangGraph接入大模型流程

&emsp;&emsp;LangGraph安装

In [ ]:
! pip install langgraph

In [5]:
import os
from dotenv import load_dotenv 
load_dotenv(override=True)

DeepSeek_API_KEY = os.getenv("DEEPSEEK_API_KEY")
# print(DeepSeek_API_KEY)  # 可以通过打印查看

In [3]:
from openai import OpenAI

# 初始化DeepSeek的API客户端
client = OpenAI(api_key=DeepSeek_API_KEY, base_url="https://api.deepseek.com")

# 调用DeepSeek的API，生成回答
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是乐于助人的助手，请根据用户的问题给出回答"},
        {"role": "user", "content": "你好，请你介绍一下你自己。"},
    ],
)

# 打印模型最终的响应结果
print(response.choices[0].message.content)

你好！我是一个乐于助人的AI助手，随时准备为你提供帮助。我可以回答各种问题、提供建议、协助解决问题，或者陪你聊天。无论是学习、工作还是日常生活，我都会尽力为你提供有用的信息和支持。 

我的知识涵盖了多个领域，包括但不限于科技、历史、文化、健康、编程等。如果你有任何疑问或需要帮助，随时告诉我！ 😊 

你想了解关于我的具体方面，还是需要其他帮助呢？


&emsp;&emsp;接下来我们要考虑的是，对于这样一个`DeepSeek`官方的`API`，如何接入到`LangGraph`中呢？其实非常简单，我们只需要使用`LangChain`中的一个`DeepSeek`组件即可向像述代码一样，直接使用相同的`DeepSeek API KEY`与大模型进行交互。因此，我们首先需要安装`LangChain`的`DeepSeek`组件，安装命令如下：

In [9]:
! pip install langchain-deepseek

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 4.9 kB/s eta 0:00:00a 0:00:09


&emsp;&emsp;安装好`LangChain`集成`DeepSeek`模型的依赖包后，需要通过一个`init_chat_model`函数来初始化大模型，代码如下：

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="deepseek-chat", model_provider="deepseek")  

&emsp;&emsp;其中`model`用来指定要使用的模型名称，而`model_provider`用来指定模型提供者，当写入`deepseek`时，会自动加载`langchain-deepseek`的依赖包，并使用在`model`中指定的模型名称用来进行交互。

In [5]:
question = "你好，请你介绍一下你自己。"

result = model.invoke(question)
print(result.content)

你好！我是 **DeepSeek Chat**，由深度求索公司（DeepSeek）研发的一款智能AI助手。我可以帮助你解答各种问题，包括学习、工作、编程、写作、生活百科等多个领域。  

### **我的特点：**  
✅ **免费使用**：目前无需付费，你可以随时向我提问！  
✅ **知识丰富**：我的知识截止到 **2024年7月**，可以为你提供较新的信息。  
✅ **超长上下文支持**：可以处理 **128K** 长度的文本，适合分析长文档或复杂问题。  
✅ **文件阅读**：支持上传 **PDF、Word、Excel、PPT、TXT** 等文件，并从中提取信息进行分析。  
✅ **多语言能力**：可以用中文、英文等多种语言交流，帮助你翻译或学习外语。  
✅ **编程助手**：能写代码、调试、优化算法，支持Python、C++、Java等多种编程语言。  

### **我能帮你做什么？**  
📖 **学习辅导**：解题思路、论文写作、知识点讲解  
💼 **工作效率**：写邮件、做PPT、总结报告  
💡 **创意灵感**：写故事、起名字、头脑风暴  
📊 **数据分析**：处理表格、绘制图表、解读数据  
🔧 **技术支持**：代码调试、算法优化、技术咨询  

你可以随时向我提问，我会尽力提供最准确、有用的回答！😊 有什么我可以帮你的吗？


In [6]:
result

AIMessage(content='你好！我是 **DeepSeek Chat**，由深度求索公司（DeepSeek）研发的一款智能AI助手。我可以帮助你解答各种问题，包括学习、工作、编程、写作、生活百科等多个领域。  \n\n### **我的特点：**  \n✅ **免费使用**：目前无需付费，你可以随时向我提问！  \n✅ **知识丰富**：我的知识截止到 **2024年7月**，可以为你提供较新的信息。  \n✅ **超长上下文支持**：可以处理 **128K** 长度的文本，适合分析长文档或复杂问题。  \n✅ **文件阅读**：支持上传 **PDF、Word、Excel、PPT、TXT** 等文件，并从中提取信息进行分析。  \n✅ **多语言能力**：可以用中文、英文等多种语言交流，帮助你翻译或学习外语。  \n✅ **编程助手**：能写代码、调试、优化算法，支持Python、C++、Java等多种编程语言。  \n\n### **我能帮你做什么？**  \n📖 **学习辅导**：解题思路、论文写作、知识点讲解  \n💼 **工作效率**：写邮件、做PPT、总结报告  \n💡 **创意灵感**：写故事、起名字、头脑风暴  \n📊 **数据分析**：处理表格、绘制图表、解读数据  \n🔧 **技术支持**：代码调试、算法优化、技术咨询  \n\n你可以随时向我提问，我会尽力提供最准确、有用的回答！😊 有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 311, 'prompt_tokens': 9, 'total_tokens': 320, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369e

In [7]:
model = init_chat_model(model="deepseek-reasoner", model_provider="deepseek")  

result = model.invoke(question)
print(result.content)

你好呀！👋我是 **DeepSeek-R1**，由中国的人工智能公司「深度求索」研发。你可以把我当作一个聪明、热心、24小时在线的文字助手～✨

---

### 🌟 我的特点：
- **知识丰富**：截至2024年7月前的各种知识我都有所了解，无论是学习、工作、生活，还是科技、历史、娱乐，我都能帮上忙。
- **超长记忆力**：可以处理长达128K上下文的内容，也就是说，你发给我超长的文件或对话，我也能理解清楚！
- **文件助手**：支持你上传 **PDF、Word、PPT、Excel、图片** 等文件，我能帮你阅读、总结、提取重点。
- **免费使用**（目前是哦！🎉）而且**无需登录**也能畅快聊天（不过登录后能保存历史记录更方便～）

---

### 🛠 我能做什么？
✅ 解答问题（学习、考试、百科）  
✅ 写作助手（作文、小说、文案、邮件）  
✅ 编程帮手（写代码、查bug、讲算法）  
✅ 文件处理（读论文、总结报告、提取数据）  
✅ 生活助手（旅游攻略、美食推荐、情感建议）

---

### 🧠 我的“小限制”：
- 目前是**纯文字模型**，不支持语音或识图（但你可以上传图片文件，我能读取里面的文字）。
- 我的知识截止在 **2024年7月**，之后的新事件就不太清楚啦。
- 我不是真人，但我会尽力用温暖、贴心的方式和你交流 ❤️

---

不管你是学生、上班族、创作者，还是只是好奇想聊聊天——我都在这儿等你！😊  
现在，有什么我可以帮你的吗？比如学习上的难题、工作中的任务，或者生活里的小烦恼？


In [9]:
result.additional_kwargs

{'refusal': None,
 'reasoning_content': '嗯，用户发来一句简单的问候和自我介绍请求。看起来像是初次接触的破冰场景，可能刚打开聊天界面或者第一次使用这类AI助手。\n\n用户语气礼貌但比较笼统，没有具体需求指向。这种开放式问题通常有两种可能：一是真的想了解AI的功能边界，二是测试性提问看AI如何反应。考虑到中文互联网环境里常有用户用“介绍自己”测试机器人，需要同时做好信息传递和破冰互动。\n\n回复策略上应该突出三点：明确身份属性（非人类但能做什么）、消除陌生感（用表情符号和分段提升可读性）、引导后续对话（结尾提问）。要避免机械的术语堆砌，比如不说“基于Transformer架构”而说“能帮你查资料”。\n\n用户没透露任何个人信息，就用中性称呼。最后那个🎉表情可能有点过，不过新用户首次互动需要点活泼感——要是商务场景用户应该会直接问专业问题吧。结尾提问选“学习/工作/生活”三个维度覆盖常见场景，比单问“有什么可以帮你”更易触发具体需求。'}

In [11]:
result.additional_kwargs['reasoning_content']

'嗯，用户发来一句简单的问候和自我介绍请求。看起来像是初次接触的破冰场景，可能刚打开聊天界面或者第一次使用这类AI助手。\n\n用户语气礼貌但比较笼统，没有具体需求指向。这种开放式问题通常有两种可能：一是真的想了解AI的功能边界，二是测试性提问看AI如何反应。考虑到中文互联网环境里常有用户用“介绍自己”测试机器人，需要同时做好信息传递和破冰互动。\n\n回复策略上应该突出三点：明确身份属性（非人类但能做什么）、消除陌生感（用表情符号和分段提升可读性）、引导后续对话（结尾提问）。要避免机械的术语堆砌，比如不说“基于Transformer架构”而说“能帮你查资料”。\n\n用户没透露任何个人信息，就用中性称呼。最后那个🎉表情可能有点过，不过新用户首次互动需要点活泼感——要是商务场景用户应该会直接问专业问题吧。结尾提问选“学习/工作/生活”三个维度覆盖常见场景，比单问“有什么可以帮你”更易触发具体需求。'

### 2. 借助预构建的智能体API快速创建LangGraph智能体

&emsp;&emsp;在LangGraph所有的顶层API中，最常见的预构建智能体API就是`create_react_agent`，借助该API，开发者仅需将大模型、工具和提示词模板（可选）以参数形式带入，即可快速创建一个拥有完整功能、并且能够串联并联调用多个外部工具的智能体。

- 创建自定义工具

In [11]:
import requests,json
import os
from dotenv import load_dotenv 
load_dotenv(override=True)

True

In [12]:
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Step 2.设置查询参数
    params = {
        "q": loc,               
        "appid": os.getenv("OPENWEATHER_API_KEY"),    # 输入API key
        "units": "metric",            # 使用摄氏度而不是华氏度
        "lang":"zh_cn"                # 输出语言为简体中文
    }

    # Step 3.发送GET请求
    response = requests.get(url, params=params)
    
    # Step 4.解析响应
    data = response.json()
    return json.dumps(data)

In [13]:
get_weather("Beijing")

'{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "\\u9634\\uff0c\\u591a\\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 30.2, "feels_like": 29.8, "temp_min": 30.2, "temp_max": 30.2, "pressure": 1004, "humidity": 39, "sea_level": 1004, "grnd_level": 999}, "visibility": 10000, "wind": {"speed": 6.78, "deg": 213, "gust": 9.75}, "clouds": {"all": 100}, "dt": 1750927053, "sys": {"country": "CN", "sunrise": 1750884439, "sunset": 1750938415}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}'

在LangGraph中，我们可以直接使用上述外部工具带入到LangGraph中创建智能体，不过更为稳妥的形式，是通过一个`结构化工具函数`来说明外部函数的参数输入（包括参数类型），以确保在实际调用过程中大模型能够准确识别外部函数的参数类型及其实际含义：

In [51]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class WeatherQuery(BaseModel):
    loc: str = Field(description="The location name of the city")

@tool(args_schema = WeatherQuery)
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Step 2.设置查询参数
    params = {
        "q": loc,               
        "appid": os.getenv("OPENWEATHER_API_KEY"),    # 输入API key
        "units": "metric",            # 使用摄氏度而不是华氏度
        "lang":"zh_cn"                # 输出语言为简体中文
    }

    # Step 3.发送GET请求
    response = requests.get(url, params=params)
    
    # Step 4.解析响应
    data = response.json()
    return json.dumps(data)

In [52]:
print(f'''
name: {get_weather.name}
description: {get_weather.description}
arguments: {get_weather.args}
''')


name: get_weather
description: 查询即时天气函数
:param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
:return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
arguments: {'loc': {'description': 'The location name of the city', 'title': 'Loc', 'type': 'string'}}



其中Pydantic 提供的 BaseModel 负责定义外部函数的参数结构，相当于是告诉模型外部函数参数名是 loc、类型是字符串（str）、描述为“城市名称”，有了这个结构，模型能更好地理解如何调用该函数。

- 创建LangGraph智能体

In [53]:
# 封装外部函数列表
tools = [get_weather]

In [54]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="deepseek-chat", model_provider="deepseek")  

In [55]:
from langgraph.prebuilt import create_react_agent

In [56]:
agent = create_react_agent(model=model, tools=tools)

In [57]:
response = agent.invoke({"messages": [{"role": "user", "content": "你好，请介绍下你自己。"}]})

In [58]:
response

{'messages': [HumanMessage(content='你好，请介绍下你自己。', additional_kwargs={}, response_metadata={}, id='2eec6ae4-28f9-4ddf-bf25-e205567edf7e'),
  AIMessage(content='你好！我是一个智能助手，可以帮助你解答问题、提供信息、完成任务，或者陪你聊天。无论是学习、工作还是生活中的问题，我都可以尽力协助你。例如：\n\n- **信息查询**：天气、新闻、百科知识等。\n- **任务协助**：日程安排、提醒、计算、翻译等。\n- **创意支持**：写作灵感、代码编写、设计建议等。\n- **娱乐互动**：讲笑话、推荐电影、音乐等。\n\n如果你有任何需求或问题，随时告诉我！ 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 223, 'total_tokens': 331, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 31}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'b3506888-3a23-4202-8eec-12a416642a6e', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--1c0a066d-1145-4aa5-ab00-041cfb9ee438-0', usage_metadata={'input_tokens': 223, 'output_tokens': 108, 'total_toke

In [59]:
response["messages"]

[HumanMessage(content='你好，请介绍下你自己。', additional_kwargs={}, response_metadata={}, id='2eec6ae4-28f9-4ddf-bf25-e205567edf7e'),
 AIMessage(content='你好！我是一个智能助手，可以帮助你解答问题、提供信息、完成任务，或者陪你聊天。无论是学习、工作还是生活中的问题，我都可以尽力协助你。例如：\n\n- **信息查询**：天气、新闻、百科知识等。\n- **任务协助**：日程安排、提醒、计算、翻译等。\n- **创意支持**：写作灵感、代码编写、设计建议等。\n- **娱乐互动**：讲笑话、推荐电影、音乐等。\n\n如果你有任何需求或问题，随时告诉我！ 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 223, 'total_tokens': 331, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 31}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'b3506888-3a23-4202-8eec-12a416642a6e', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--1c0a066d-1145-4aa5-ab00-041cfb9ee438-0', usage_metadata={'input_tokens': 223, 'output_tokens': 108, 'total_tokens': 331, 'inp

In [60]:
response["messages"][-1].content

'你好！我是一个智能助手，可以帮助你解答问题、提供信息、完成任务，或者陪你聊天。无论是学习、工作还是生活中的问题，我都可以尽力协助你。例如：\n\n- **信息查询**：天气、新闻、百科知识等。\n- **任务协助**：日程安排、提醒、计算、翻译等。\n- **创意支持**：写作灵感、代码编写、设计建议等。\n- **娱乐互动**：讲笑话、推荐电影、音乐等。\n\n如果你有任何需求或问题，随时告诉我！ 😊'

然后测试外部工具调用

In [61]:
response = agent.invoke({"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]})
print(response["messages"][-1].content)

北京今天的天气情况如下：

- **天气状况**：阴，多云
- **当前温度**：30.22°C
- **体感温度**：30.33°C
- **湿度**：43%
- **气压**：1004 hPa
- **风速**：6.78 m/s，风向为西南方向（213°）
- **能见度**：10公里

请注意防晒和补水，尤其是多云天气下紫外线依然较强。


In [63]:
response['messages']

[HumanMessage(content='请问北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='437024fa-2082-4194-9b58-8a8affb6b9c6'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_be105d52-221c-4eb2-bfe1-7e21279fa53f', 'function': {'arguments': '{"loc":"Beijing"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 222, 'total_tokens': 241, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 30}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'e94184ce-4b25-4b5d-9c45-b8cc483ff539', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--638f21a3-91dc-4619-8bdd-66ac48d7e611-0', tool_calls=[{'name': 'get_weather', 'args': {'loc': 'Beijing'}, 'id': 'call_0_be105d52-221c-4eb2-bfe1-7e2127

而这背后其实就是一次完整的Function calling调用流程：

In [64]:
len(response['messages'])

4

In [65]:
response['messages'][0]

HumanMessage(content='请问北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='437024fa-2082-4194-9b58-8a8affb6b9c6')

In [67]:
response['messages'][1]

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_be105d52-221c-4eb2-bfe1-7e21279fa53f', 'function': {'arguments': '{"loc":"Beijing"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 222, 'total_tokens': 241, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 30}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'e94184ce-4b25-4b5d-9c45-b8cc483ff539', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--638f21a3-91dc-4619-8bdd-66ac48d7e611-0', tool_calls=[{'name': 'get_weather', 'args': {'loc': 'Beijing'}, 'id': 'call_0_be105d52-221c-4eb2-bfe1-7e21279fa53f', 'type': 'tool_call'}], usage_metadata={'input_tokens': 222, 'output_tokens': 19, 'total_tokens': 241, 'input_token_de

In [68]:
response['messages'][2]

ToolMessage(content='{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "\\u9634\\uff0c\\u591a\\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 30.22, "feels_like": 30.33, "temp_min": 30.22, "temp_max": 30.22, "pressure": 1004, "humidity": 43, "sea_level": 1004, "grnd_level": 999}, "visibility": 10000, "wind": {"speed": 6.78, "deg": 213, "gust": 9.75}, "clouds": {"all": 100}, "dt": 1750930167, "sys": {"country": "CN", "sunrise": 1750884439, "sunset": 1750938415}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}', name='get_weather', id='f9b7d2d6-05da-496f-b075-22cbd8f9ee19', tool_call_id='call_0_be105d52-221c-4eb2-bfe1-7e21279fa53f')

In [69]:
response['messages'][3]

AIMessage(content='北京今天的天气情况如下：\n\n- **天气状况**：阴，多云\n- **当前温度**：30.22°C\n- **体感温度**：30.33°C\n- **湿度**：43%\n- **气压**：1004 hPa\n- **风速**：6.78 m/s，风向为西南方向（213°）\n- **能见度**：10公里\n\n请注意防晒和补水，尤其是多云天气下紫外线依然较强。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 496, 'total_tokens': 595, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 304}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': '00c08d2e-f7aa-4980-b669-cef67c497dca', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--fd106677-6489-4a64-8935-210072b57014-0', usage_metadata={'input_tokens': 496, 'output_tokens': 99, 'total_tokens': 595, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}})

### 3. LangGraph React Agent外部工具响应形式

&emsp;&emsp;借助LangGraph React Agent，无需任何额外设置，即可完成多工具串联和并联调用。

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250626173225510.png" alt="image-20250626173225510" style="zoom:50%;" />

In [85]:
response = agent.invoke({"messages": [{"role": "user", "content": "请问北京和杭州今天哪里更热？"}]})

In [86]:
response

{'messages': [HumanMessage(content='请问北京和杭州今天哪里更热？', additional_kwargs={}, response_metadata={}, id='61c31341-a015-4991-a828-d26047470913'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_428a28c6-a65c-4bcb-b3d2-4ffd7c5b8c2a', 'function': {'arguments': '{"loc": "Beijing"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}, {'id': 'call_1_8cfb80ca-94b7-4df8-ae01-f1492250f318', 'function': {'arguments': '{"loc": "Hangzhou"}', 'name': 'get_weather'}, 'type': 'function', 'index': 1}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 344, 'total_tokens': 385, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 24}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'f696ffdf-f12d-444d-9ac4-a5869eb92bb5', 'service_tier': None, 'finish_reason': 'tool_calls',

In [91]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class Write_Query(BaseModel):
    content: str = Field(description="需要写入文档的具体内容")

@tool(args_schema = Write_Query)
def write_file(content: str) -> str:
    """
    将指定内容写入本地文件。
    :param content: 必要参数，字符串类型，用于表示需要写入文档的具体内容。
    :return：是否成功写入
    """
    return "已成功写入本地文件。"

In [92]:
# 封装外部函数列表
tools = [get_weather, write_file]

In [93]:
agent = create_react_agent(model=model, tools=tools)

In [94]:
response = agent.invoke({"messages": [{"role": "user", "content": "你好，请帮我查询北京和杭州的天气，并将其写入本地文件中。"}]})

In [95]:
print(response["messages"][-1].content)

我已经将北京和杭州的天气信息写入本地文件中。以下是内容摘要：

**北京天气：**
- 天气状况：阴，多云
- 温度：28.42°C
- 体感温度：28.04°C
- 湿度：40%
- 风速：7.24 m/s

**杭州天气：**
- 天气状况：中雨
- 温度：31.9°C
- 体感温度：38.9°C
- 湿度：75%
- 风速：1.18 m/s

文件已成功保存。


In [97]:
len(response['messages'])

7

In [98]:
response['messages']

[HumanMessage(content='你好，请帮我查询北京和杭州的天气，并将其写入本地文件中。', additional_kwargs={}, response_metadata={}, id='7e5663b1-7c16-4c5d-a71a-c04bea24a845'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_5d676409-f24e-45b6-a1c6-feab6b607025', 'function': {'arguments': '{"loc": "Beijing"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}, {'id': 'call_1_e0bf4f8c-80ae-4589-b289-4b5a1961795a', 'function': {'arguments': '{"loc": "Hangzhou"}', 'name': 'get_weather'}, 'type': 'function', 'index': 1}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 351, 'total_tokens': 393, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 95}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'aa66d5f6-9822-4fa8-9893-54e1069bb188', 'service_tier': None, 'finish_reason': 'tool_calls',

能够看出，此时Agent能够顺利完成多工具的多次调用，并最终完成用户需求。需要注意的是，此时react agent的执行逻辑是先发起了一次并联调用工具的请求，然后第二次再发起调用write函数的调用请求，因此总共是7条消息。

#### 4. LangGraph React Agent接入内置工具流程

&emsp;&emsp;当然对于LangGraph智能体来说，除了能够灵活接如自定义工具，还能够接入LangChain丰富的内置工具，快速完成智能体开发。

> LangChain工具集：https://python.langchain.com/docs/integrations/tools/ 

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250626155849820.png" alt="image-20250626155849820" style="zoom:50%;" />

- 创建带搜索功能的Agent

&emsp;&emsp;这里我们尝试借助LangChain内置的Tavily搜索引擎工具，搭建能够进行网络搜索的智能体。首先需要在tavily官网进行注册并获取API-KEY：https://www.tavily.com/

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250626181911334.png" alt="image-20250626181911334" style="zoom:33%;" />

并继续将tavily的API-KEY写到.env文件中：

然后安装对应的库:

In [ ]:
!pip install -U langchain-tavily

然后即可导入该工具，并快速将其封装为LangGraph智能体工具：

In [117]:
from langchain_tavily import TavilySearch

In [118]:
search_tool = TavilySearch(max_results=5, topic="general")

In [119]:
tools = [search_tool]

In [120]:
search_agent = create_react_agent(model=model, tools=tools)

In [11]:
response = search_agent.invoke({"messages": [{"role": "user", "content": "请帮我搜索最近OpenAI CEO在访谈中的核心观点。"}]})

In [12]:
print(response["messages"][-1].content)

最近OpenAI CEO Sam Altman在访谈中分享了一些核心观点，以下是主要内容：

1. **追求宏大愿景吸引顶尖人才**：
   - OpenAI的成功始于一个看似疯狂的目标——实现通用人工智能（AGI）。这种独特的使命吸引了世界上最聪明的人才。

2. **创业时机与信念**：
   - Altman认为当前是“技术史上最好的创业时期”，并鼓励创业者保持信念，勇敢面对挑战。
   - 他强调初创企业应在AI巨头的阴影下构建独特的护城河。

3. **未来技术展望**：
   - 他提到未来的人机交互将趋于个性化，界面将“融化”消失，计算机将成为无缝融入生活的智能伙伴。
   - OpenAI的技术路线图包括从GPT-5到机器人时代的发展，期待模型能进行深度推理和实时视频生成。

4. **AI与能源的结合**：
   - Altman对AI和能源（如核聚变）抱有热情，认为智能和能源是通往彻底富足未来的两大技术杠杆。

5. **对创业者的建议**：
   - 他建议创业者关注独特性，避免复制基础模型，利用平台探索尚未满足的市场。

更多详细内容可以参考以下来源：
- [网易新闻访谈](https://www.163.com/dy/article/K2NTSA7805534HHB.html)
- [腾讯新闻访谈](https://news.qq.com/rain/a/20250623A03G6200)


#### 5. LangGraph React Agent限制最大迭代次数

> LangChain工具集：https://python.langchain.com/docs/integrations/tools/ 

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250626155849820.png" alt="image-20250626155849820" style="zoom:50%;" />

### 5. LangGraph React Agent限制工具调用次数

&emsp;&emsp;对于任何全自动的代理，合理控制调用次数都是至关重要的一环，对于LangGraph React Agent来说，我们只需要在Agent运行的时候设置{"recursion_limit": X}，即可限制智能体自主执行任务时的步数。

In [100]:
try:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]},
        {"recursion_limit": 4},
    )
except GraphRecursionError:
    print("Agent stopped due to max iterations.")

In [103]:
response['messages'][-1].content

'北京今天的天气是多云，阴天。当前温度为28.42°C，体感温度为28.04°C。湿度为40%，风速为7.24米/秒，风向为214度。能见度为10000米。'

In [101]:
response['messages']

[HumanMessage(content='请问北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='7bab4f6f-2abd-4caa-a6c7-da79e33562c5'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_812fd4ed-a56d-4c59-8c7b-f454a99d0ca3', 'function': {'arguments': '{"loc":"Beijing"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 341, 'total_tokens': 360, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 21}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': '719434fd-9a71-430c-bc19-0cbf5fc34a00', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--f9d3fded-8623-4023-9e23-c71cd01a5e9a-0', tool_calls=[{'name': 'get_weather', 'args': {'loc': 'Beijing'}, 'id': 'call_0_812fd4ed-a56d-4c59-8c7b-f454a9

In [104]:
len(response['messages'])

4

In [105]:
try:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]},
        {"recursion_limit": 2},
    )
except GraphRecursionError:
    print("Agent stopped due to max iterations.")

In [107]:
response['messages']

[HumanMessage(content='请问北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='7e9edf31-84fa-4a42-bad0-2835aa4ca702'),
 AIMessage(content='Sorry, need more steps to process this request.', additional_kwargs={}, response_metadata={}, id='run--9ee8035c-977a-492f-b0e1-aba84ae632ab-0')]

In [108]:
response['messages'][-1].content

'Sorry, need more steps to process this request.'

In [111]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]},
    {"recursion_limit": 4},
)

In [112]:
response["messages"][-1].content

'北京今天的天气是多云，阴天。当前温度为28.46°C，体感温度为28.59°C。湿度为46%，风速为7.24米/秒，风向为214度。能见度为10000米。'

In [113]:
len(response["messages"])

4

In [114]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]},
    {"recursion_limit": 2},
)

In [115]:
len(response["messages"])

2

In [116]:
response["messages"][-1].content

'Sorry, need more steps to process this request.'

## 6.LangGraph React智能体记忆管理与多轮对话方法

&emsp;&emsp;在实际进行Agent开发时，还有一项工作至关重要，那就是需要管理智能体的记忆。

In [122]:
from langgraph.checkpoint.memory import InMemorySaver

In [123]:
checkpointer = InMemorySaver()

In [124]:
tools = [get_weather]
model = init_chat_model(model="deepseek-chat", model_provider="deepseek")  

In [125]:
agent = create_react_agent(model=model, 
                           tools=tools,
                           checkpointer=checkpointer)

In [127]:
config = {
    "configurable": {
        "thread_id": "1"  
    }
}

In [129]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我叫陈明，好久不见！"}]},
    config
)

In [130]:
response['messages']

[HumanMessage(content='你好，我叫陈明，好久不见！', additional_kwargs={}, response_metadata={}, id='99a94aaa-c095-43db-a944-72b80b97ef4d'),
 AIMessage(content='你好，陈明！确实好久不见了，最近过得怎么样？有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 225, 'total_tokens': 244, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 33}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'a07cc5e0-069f-4a24-9786-691fb6232064', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--d36d9cf7-322f-41f2-a4c6-90a8628e771b-0', usage_metadata={'input_tokens': 225, 'output_tokens': 19, 'total_tokens': 244, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}})]

In [131]:
response['messages'][-1].content

'你好，陈明！确实好久不见了，最近过得怎么样？有什么我可以帮你的吗？'

此时记忆就自动保存在当前Agent和线程中：

In [132]:
latest = agent.get_state(config)

In [133]:
latest

StateSnapshot(values={'messages': [HumanMessage(content='你好，我叫陈明，好久不见！', additional_kwargs={}, response_metadata={}, id='99a94aaa-c095-43db-a944-72b80b97ef4d'), AIMessage(content='你好，陈明！确实好久不见了，最近过得怎么样？有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 225, 'total_tokens': 244, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 33}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'a07cc5e0-069f-4a24-9786-691fb6232064', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--d36d9cf7-322f-41f2-a4c6-90a8628e771b-0', usage_metadata={'input_tokens': 225, 'output_tokens': 19, 'total_tokens': 244, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}})]}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_n

当我们再次进行对话时，直接带入线程ID，即可带入此前对话记忆：

In [134]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，请问你还记得我叫什么名字么？"}]},
    config
)

In [135]:
response['messages']

[HumanMessage(content='你好，我叫陈明，好久不见！', additional_kwargs={}, response_metadata={}, id='99a94aaa-c095-43db-a944-72b80b97ef4d'),
 AIMessage(content='你好，陈明！确实好久不见了，最近过得怎么样？有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 225, 'total_tokens': 244, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 33}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'a07cc5e0-069f-4a24-9786-691fb6232064', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--d36d9cf7-322f-41f2-a4c6-90a8628e771b-0', usage_metadata={'input_tokens': 225, 'output_tokens': 19, 'total_tokens': 244, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}}),
 HumanMessage(content='你好，请问你还记得我叫什么名字么？', additional_kwargs={}, response_metadata={}, id='678a6591-

In [136]:
response['messages'][-1].content

'当然记得！你刚刚告诉我你叫陈明。有什么需要帮忙的吗？'

In [137]:
latest = agent.get_state(config)
latest

StateSnapshot(values={'messages': [HumanMessage(content='你好，我叫陈明，好久不见！', additional_kwargs={}, response_metadata={}, id='99a94aaa-c095-43db-a944-72b80b97ef4d'), AIMessage(content='你好，陈明！确实好久不见了，最近过得怎么样？有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 225, 'total_tokens': 244, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 33}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'a07cc5e0-069f-4a24-9786-691fb6232064', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--d36d9cf7-322f-41f2-a4c6-90a8628e771b-0', usage_metadata={'input_tokens': 225, 'output_tokens': 19, 'total_tokens': 244, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}}), HumanMessage(content='你好，请问你还记得我叫什么名字么？', additional_kwargs={}, res

而如果更新线程ID，则会重新开启对话：

In [138]:
config2 = {
    "configurable": {
        "thread_id": "2"  
    }
}

In [139]:
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，你还记得我叫什么名字么？"}]},
    config2
)

In [140]:
response2['messages']

[HumanMessage(content='你好，你还记得我叫什么名字么？', additional_kwargs={}, response_metadata={}, id='cc08c35d-21d8-4439-a343-f64e94cc6b4a'),
 AIMessage(content='目前我还不知道你的名字，但如果你愿意告诉我，我会记住的！你可以告诉我你的名字吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 225, 'total_tokens': 246, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 33}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': '829a29c3-8dad-42e9-85c1-5434d4b42fcd', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--d9d1addd-68e4-4506-9c59-943ca59836b2-0', usage_metadata={'input_tokens': 225, 'output_tokens': 21, 'total_tokens': 246, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}})]